In [1]:
!pip install groq python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 9.1 MB/s eta 0:00:00


In [12]:
#PES2UG23CS647
import os
from groq import Groq
from getpass import getpass

# Secure input (hidden while typing)
groq_key = getpass("Enter your GROQ API Key: ")

# Store temporarily in environment
os.environ["GROQ_API_KEY"] = groq_key

# Initialize client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

BASE_MODEL = "mixtral-8x7b-32768"

print("Groq client initialized successfully.")

Enter your GROQ API Key: ··········
Groq client initialized successfully.


In [13]:
# Fast model for routing (cheap + deterministic)
ROUTER_MODEL = "llama-3.1-8b-instant"

# Powerful model for expert responses
EXPERT_MODEL = "llama-3.3-70b-versatile"

In [14]:
#PES2UG23CS647
MODEL_CONFIG = {
    "technical": {
        "system_prompt": (
            "You are a Senior Technical Support Engineer. "
            "Be precise, structured, and code-focused. "
            "Provide debugging steps and corrected code when applicable."
        ),
        "temperature": 0.7,
    },
    "billing": {
        "system_prompt": (
            "You are a Billing Support Specialist. "
            "Be empathetic, professional, and policy-driven. "
            "Clearly explain refund rules and next steps."
        ),
        "temperature": 0.7,
    },
    "general": {
        "system_prompt": (
            "You are a friendly and helpful assistant. "
            "Respond conversationally and clearly."
        ),
        "temperature": 0.7,
    },
}

In [15]:
def route_prompt(user_input: str) -> str:
    routing_prompt = (
        "Classify the following text into one of these categories:\n"
        "[technical, billing, general, tool]\n\n"
        "If the user asks for live data such as crypto prices, "
        "stock prices, or real-time information, classify as 'tool'.\n\n"
        "Return ONLY the category name.\n\n"
        f"Text: {user_input}"
    )

    response = client.chat.completions.create(
        model=ROUTER_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a strict classifier."},
            {"role": "user", "content": routing_prompt},
        ],
    )

    return response.choices[0].message.content.strip().lower()

In [16]:
#PES2UG23CS647
def fetch_bitcoin_price():
    # Mock real-time fetch
    return "The current price of Bitcoin is $64,250 (mock value)."

In [17]:
def process_request(user_input: str) -> str:

    # Step 1: Route the query
    category = route_prompt(user_input)
    print(f"[Router Decision]: {category}")

    # Step 2: Tool Handling
    if category == "tool":
        return fetch_bitcoin_price()

    # Step 3: Fallback safety
    if category not in MODEL_CONFIG:
        category = "general"

    expert_config = MODEL_CONFIG[category]

    # Step 4: Call Expert Model
    response = client.chat.completions.create(
        model=EXPERT_MODEL,
        temperature=expert_config["temperature"],
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input},
        ],
    )

    return response.choices[0].message.content

In [18]:
query = input("Enter your question: ")
answer = process_request(query)

print("\nAssistant:", answer)

Enter your question: I was charged twice for my subscription this month
[Router Decision]: billing

Assistant: I'm so sorry to hear that you were charged twice for your subscription this month. I can imagine how frustrating that must be for you. I'm here to help you resolve this issue as quickly as possible.

To initiate the refund process, I'll need to verify some information with you. Could you please provide me with your subscription details, including your account name and the date of the duplicate charge? This will help me to locate the issue and expedite the refund process.

Additionally, I want to inform you that our company has a strict policy of refunding duplicate charges within 3-5 business days of notification. Please be assured that we will do our best to rectify the situation as soon as possible.

If the duplicate charge is indeed an error on our part, we will provide a full refund for the incorrect charge. You will receive an email notification once the refund has been p